# Day 2 — NumPy + Matplotlib
### TensorOrbit EDA Bootcamp | 22 August 2026

This notebook is built for **live-coding** — type it out with the class rather than just running it. Markdown cells are your talking points; code cells are what you type/run live.

**Dataset:** `sustainability_cities.csv` — same theme as Day 1, so today plainly continues yesterday's work.

## Recap of Day 1 (5 min)

Remind the class:
- Yesterday we used **Pandas** to load, clean, and explore a sustainability dataset — `.head()`, `.info()`, `.groupby()`, filtering, handling missing values.
- Today we go **underneath** Pandas to **NumPy** (the number-crunching engine Pandas is built on), then **Matplotlib** to turn numbers into pictures.
- By the end of today, you'll take a question -> find the answer with NumPy -> show it as a chart.

Quickly resolve any setup issues before continuing.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('sustainability_cities.csv')
df.head()

---
# Part 1: NumPy

## Why NumPy? (talk before code)
Ask: *"If I wanted to double every number in a list of a million values, how would you do it in plain Python?"* -> a for-loop.

NumPy does this **without an explicit loop** (vectorisation) — it's written in C under the hood, so it's dramatically faster and the code is shorter. Pandas columns are actually NumPy arrays underneath — this is *why* Pandas can do fast math on entire columns at once.

## Creating arrays

In [ ]:
a = np.array([1, 2, 3, 4, 5])
print(a)

zeros = np.zeros(5)
ones = np.ones((2, 3))
range_arr = np.arange(0, 10, 2)

print(zeros)
print(ones)
print(range_arr)

**Talking point:** connect straight to the dataset — pull a real column out as a NumPy array.

In [ ]:
co2 = df['co2_emissions_tons_per_capita'].to_numpy()
co2[:10]

## Indexing & slicing (1D and 2D)

In [ ]:
print(co2[0])       # first value
print(co2[:5])      # first 5
print(co2[-3:])     # last 3

grid = np.array([[1,2,3],[4,5,6],[7,8,9]])
print(grid[1, 2])    # row 1, col 2
print(grid[:, 0])    # whole first column
print(grid[0, :])    # whole first row

## Reshaping arrays

In [ ]:
flat = np.arange(12)
print(flat)
reshaped = flat.reshape(3, 4)
print(reshaped)

## Vectorised math (no loops!)

In [ ]:
recycle = df['recycling_rate_pct'].to_numpy()
renewable = df['renewable_energy_pct'].to_numpy()

combined_score = (recycle + renewable) / 2
print(combined_score[:10])

## Boolean / fancy indexing, np.where()

**Talking point:** this is the NumPy engine behind Pandas boolean filtering they learned yesterday (`df[df['x'] > 5]`).

In [ ]:
high_co2_mask = co2 > 3
print(high_co2_mask[:10])
print(co2[high_co2_mask][:10])   # only the high-CO2 values

# np.where: label values instead of just filtering
labels = np.where(co2 > 3, 'high', 'low')
print(labels[:10])

## Sorting arrays

In [ ]:
print(np.sort(co2)[:5])   # 5 lowest
print(np.sort(co2)[-5:])  # 5 highest

## Statistical functions

In [ ]:
print('Mean CO2:', np.nanmean(co2))
print('Median CO2:', np.nanmedian(co2))
print('Std Dev CO2:', np.nanstd(co2))
print('Variance CO2:', np.nanvar(co2))

Note: we use `np.nanmean` / `np.nanmedian` / `np.nanstd` here instead of `np.mean` etc. because this column has missing values (`NaN`) — plain `np.mean` would return `NaN` for the whole result. Good moment to link back to yesterday's `.isnull().sum()`.

## Percentiles & quartiles

In [ ]:
q1 = np.nanpercentile(co2, 25)
q2 = np.nanpercentile(co2, 50)
q3 = np.nanpercentile(co2, 75)
print('Q1:', q1, '| Median:', q2, '| Q3:', q3)

## Outlier detection: Z-score and IQR methods

**Talking point:** *why* do we care about outliers before modelling? A single extreme value can badly distort an average or a trendline.

In [ ]:
# Z-score method
mean_co2 = np.nanmean(co2)
std_co2 = np.nanstd(co2)
z_scores = (co2 - mean_co2) / std_co2

outliers_z = df[np.abs(z_scores) > 3]
print('Outliers (Z-score > 3):')
outliers_z[['city', 'year', 'co2_emissions_tons_per_capita']]

In [ ]:
# IQR method
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers_iqr = df[(df['co2_emissions_tons_per_capita'] < lower_bound) | (df['co2_emissions_tons_per_capita'] > upper_bound)]
print('Outliers (IQR method):')
outliers_iqr[['city', 'year', 'co2_emissions_tons_per_capita']]

## Correlation: np.corrcoef()

In [ ]:
clean = df.dropna(subset=['co2_emissions_tons_per_capita', 'recycling_rate_pct'])
corr_matrix = np.corrcoef(clean['co2_emissions_tons_per_capita'], clean['recycling_rate_pct'])
print(corr_matrix)
print('Correlation coefficient:', corr_matrix[0, 1])

**Reading it:** the matrix's off-diagonal value (row 0, col 1) is the correlation between the two variables. Close to **+1** = strong positive relationship, close to **-1** = strong negative, close to **0** = weak/no linear relationship.

**Correlation vs. causation (concept only, no code):** ask the class — *if cities with more renewable energy also have less CO2, does renewable energy cause lower CO2, or could both be caused by something else (like wealth, policy, or city size)?* Correlation tells us variables move together; it never tells us *why* on its own.

---
# Part 2: Matplotlib

## Why visualise?
We just calculated means, outliers, and a correlation — but a table of numbers doesn't *land* the way a picture does. Matplotlib turns the NumPy output above into something the audience in Day 4's presentations will actually understand in 5 seconds.

## Line chart — trend over time

In [ ]:
yearly_avg = df.groupby('year')['co2_emissions_tons_per_capita'].mean()

plt.figure(figsize=(7,4))
plt.plot(yearly_avg.index, yearly_avg.values, marker='o', color='seagreen')
plt.title('Average CO2 Emissions per Capita by Year')
plt.xlabel('Year')
plt.ylabel('CO2 (tons per capita)')
plt.grid(True)
plt.show()

## Bar chart — comparing categories

In [ ]:
top10 = df.groupby('city')['renewable_energy_pct'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(8,4))
plt.bar(top10.index, top10.values, color='steelblue')
plt.title('Top 10 Cities by Renewable Energy %')
plt.xlabel('City')
plt.ylabel('Renewable Energy (%)')
plt.xticks(rotation=45, ha='right')
plt.show()

## Histogram — distribution of a single variable

In [ ]:
plt.figure(figsize=(7,4))
plt.hist(df['co2_emissions_tons_per_capita'].dropna(), bins=15, color='darkorange', edgecolor='black')
plt.title('Distribution of CO2 Emissions per Capita')
plt.xlabel('CO2 (tons per capita)')
plt.ylabel('Frequency')
plt.show()

## Scatter plot — relationship between two variables

In [ ]:
plt.figure(figsize=(7,4))
plt.scatter(df['renewable_energy_pct'], df['co2_emissions_tons_per_capita'], alpha=0.6, color='purple')
plt.title('Renewable Energy % vs CO2 Emissions per Capita')
plt.xlabel('Renewable Energy (%)')
plt.ylabel('CO2 (tons per capita)')
plt.show()

## Boxplot — spotting spread and outliers visually

**Talking point:** this is the *picture* version of the Z-score/IQR outlier work from the NumPy section — the dots beyond the whiskers are the same outliers.

In [ ]:
plt.figure(figsize=(5,5))
plt.boxplot(df['co2_emissions_tons_per_capita'].dropna(), vert=True)
plt.title('CO2 Emissions per Capita — Boxplot')
plt.ylabel('CO2 (tons per capita)')
plt.show()

## Pie chart — share of a whole

**Talking point:** pie charts work best with few categories (3-6) that sum to a meaningful whole — use sparingly.

In [ ]:
country_counts = df['country'].value_counts().head(5)

plt.figure(figsize=(6,6))
plt.pie(country_counts.values, labels=country_counts.index, autopct='%1.0f%%')
plt.title('Sample Distribution: Top 5 Countries by Record Count')
plt.show()

## Subplots — multiple charts side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(yearly_avg.index, yearly_avg.values, marker='o', color='seagreen')
axes[0].set_title('Avg CO2 by Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('CO2 (tons per capita)')

axes[1].scatter(df['renewable_energy_pct'], df['co2_emissions_tons_per_capita'], alpha=0.6, color='purple')
axes[1].set_title('Renewable % vs CO2')
axes[1].set_xlabel('Renewable Energy (%)')
axes[1].set_ylabel('CO2 (tons per capita)')

plt.tight_layout()
plt.show()

---
## Guided Practice (last ~20-25 min)

Take **2-3 findings** you generated in the NumPy section above and turn each into a clearly labelled chart, choosing the chart type that fits the question. Every chart needs: a title, axis labels, and (if more than one series) a legend.

Prompts to put on the board:
1. Which city had the highest average water usage per capita across all years? Show it.
2. Is there a relationship between green space % and recycling rate %? Show it.
3. How has the average recycling rate changed year over year? Show it.

Work in the empty cells below.

In [ ]:
# Practice 1


In [ ]:
# Practice 2


In [ ]:
# Practice 3


---
## Wrap-up (5 min)
- Quick show of hands: who found something surprising in their chart?
- Preview Day 3: tomorrow you'll combine Pandas + NumPy + Matplotlib into one full workflow on a *business* dataset, then get your capstone brief.